<style>
@import url('https://fonts.googleapis.com/css2?family=Computer+Modern&display=swap');
@import url('https://fonts.googleapis.com/css2?family=EB+Garamond:ital,wght@0,400;0,600;1,400&display=swap');
@import url('https://fonts.googleapis.com/css2?family=Crimson+Pro:ital,wght@0,400;0,600;1,400&display=swap');

/* Apply to entire notebook */
.jp-RenderedHTMLCommon p,
.jp-RenderedHTMLCommon li,
.jp-RenderedHTMLCommon td {
    font-family: 'EB Garamond', serif;
    font-size: 15px;
    line-height: 1.8;
}

.jp-RenderedHTMLCommon h1,
.jp-RenderedHTMLCommon h2,
.jp-RenderedHTMLCommon h3 {
    font-family: 'Crimson Pro', serif;
    color: #8B5CF6;         /* your violet */
    font-weight: 600;
}
</style>


# <font color="#14B8A6"> **Customer Churn | Pytabkit - Ensemble Baseline**


`Ensemble_TD_Classifier` from the `PyTabKit` library combines four strong tabular models into one — `RealMLP`, `XGBoost`, `LightGBM`, and `CatBoost` — each with pre-optimized hyperparameters out of the box (no manual tuning needed).
Instead of averaging predictions equally, it uses Caruana-style greedy weighting to automatically assign higher weights to better-performing sub-models on your data.
In this notebook, I use `3-Fold Stratified CV` with `ROC-AUC` as the evaluation metric to train `Ensemble_TD_Classifier` Model

In [1]:
!pip install pytabkit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 7.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import StratifiedKFold
from pytabkit import Ensemble_TD_Classifier
from sklearn.metrics import roc_auc_score
import warnings
import torch
import os
warnings.filterwarnings("ignore")

In [3]:
TARGET = "Churn"
ID = "id"

FOLDS = 5
CV = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
train = pd.read_csv('/kaggle/input/notebooks/mhamza0810/ps-s6e3-notebook-v2-baseline-fe-set-w-wo-origdata/train_fe_without_orig.csv')
test = pd.read_csv('/kaggle/input/notebooks/mhamza0810/ps-s6e3-notebook-v2-baseline-fe-set-w-wo-origdata/test_fe_without_orig.csv')
train_comp = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test_comp = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')
print(f"Train Data Shape | Rows {train.shape[0]}, Columns: {train.shape[1]}\n Test Data Shape | Rows: {test.shape[0]}, Columns: {test.shape[1]}")

Train Data Shape | Rows 594194, Columns: 151
 Test Data Shape | Rows: 254655, Columns: 150


In [5]:
cols_to_drop = ['dvae_0', 'dvae_1', 'dvae_2', 'dvae_3', 'dvae_4', 'dvae_5', 'dvae_6', 'dvae_7']

FEATURES = [col for col in train.columns if col != TARGET and col not in cols_to_drop]

In [6]:
X = train[FEATURES]
y = train[TARGET]
test = test[FEATURES]
cv_indices = list(CV.split(X, y))

In [7]:
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(cv_indices):
    print('#'*25)
    print(f"### Training Fold {fold+1} ###")
    print('#'*25)

    X_tr = X.iloc[train_idx]
    X_val = X.iloc[val_idx]
    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    model = Ensemble_TD_Classifier(
        n_cv=1,
        val_fraction=0.2,
        n_refit=0,
        device=DEVICE,
        n_threads=4,
        val_metric_name='1-auc_ovr',
        calibration_method='ts-mix',
        random_state=42 + fold,
        verbosity=2,
        tmp_folder=f"/kaggle/working/ensemble_cache/fold_{fold+1}"
    )

    model.fit(X_tr, y_tr)
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(test)[:, 1] / CV.n_splits

    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    print(f'--- Fold {fold+1} AUC: {fold_auc:.5f} ---')
    fold_scores.append(fold_auc)

oof_auc = roc_auc_score(y, oof_preds)
print(f'\nFold Scores : {[round(s, 5) for s in fold_scores]}')
print(f'Mean CV AUC : {np.mean(fold_scores):.5f} ± {np.std(fold_scores):.5f}')
print(f'Overall OOF AUC: {oof_auc:.5f}')

#########################
### Training Fold 1 ###
#########################
Columns classified as continuous: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Charge_Difference', 'tenure_qbin5', 'tenure_qbin10', 'tenure_qbin20', 'tenure_ebin5', 'tenure_ebin10', 'tenure_ebin20', 'tenure_round5', 'MonthlyCharges_qbin5', 'MonthlyCharges_qbin10', 'MonthlyCharges_qbin20', 'MonthlyCharges_ebin5', 'MonthlyCharges_ebin10', 'MonthlyCharges_ebin20', 'MonthlyCharges_round10', 'TotalCharges_qbin5', 'TotalCharges_qbin10', 'TotalCharges_qbin20', 'TotalCharges_ebin5', 'TotalCharges_ebin10', 'TotalCharges_ebin20', 'TotalCharges_round10', 'Charge_Difference_qbin5', 'Charge_Difference_qbin10', 'Charge_Difference_qbin20', 'Charge_Difference_ebin5', 'Charge

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/256: val 1-auc_ovr = 0.088285
Epoch 2/256: val 1-auc_ovr = 0.089158
Epoch 3/256: val 1-auc_ovr = 0.088294
Epoch 4/256: val 1-auc_ovr = 0.087558
Epoch 5/256: val 1-auc_ovr = 0.088024
Epoch 6/256: val 1-auc_ovr = 0.088048
Epoch 7/256: val 1-auc_ovr = 0.088321
Epoch 8/256: val 1-auc_ovr = 0.087601
Epoch 9/256: val 1-auc_ovr = 0.089063
Epoch 10/256: val 1-auc_ovr = 0.087949
Epoch 11/256: val 1-auc_ovr = 0.088111
Epoch 12/256: val 1-auc_ovr = 0.086029
Epoch 13/256: val 1-auc_ovr = 0.086150
Epoch 14/256: val 1-auc_ovr = 0.085443
Epoch 15/256: val 1-auc_ovr = 0.085118
Epoch 16/256: val 1-auc_ovr = 0.084258
Epoch 17/256: val 1-auc_ovr = 0.084106
Epoch 18/256: val 1-auc_ovr = 0.084187
Epoch 19/256: val 1-auc_ovr = 0.084473
Epoch 20/256: val 1-auc_ovr = 0.085181
Epoch 21/256: val 1-auc_ovr = 0.085802
Epoch 22/256: val 1-auc_ovr = 0.087551
Epoch 23/256: val 1-auc_ovr = 0.088125
Epoch 24/256: val 1-auc_ovr = 0.087590
Epoch 25/256: val 1-auc_ovr = 0.089147
Epoch 26/256: val 1-auc_ovr = 0.08

`Trainer.fit` stopped: `max_epochs=256` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Obtained ensemble weights: [16  0 19  0]
--- Fold 1 AUC: 0.91699 ---
#########################
### Training Fold 2 ###
#########################
Columns classified as continuous: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Charge_Difference', 'tenure_qbin5', 'tenure_qbin10', 'tenure_qbin20', 'tenure_ebin5', 'tenure_ebin10', 'tenure_ebin20', 'tenure_round5', 'MonthlyCharges_qbin5', 'MonthlyCharges_qbin10', 'MonthlyCharges_qbin20', 'MonthlyCharges_ebin5', 'MonthlyCharges_ebin10', 'MonthlyCharges_ebin20', 'MonthlyCharges_round10', 'TotalCharges_qbin5', 'TotalCharges_qbin10', 'TotalCharges_qbin20', 'TotalCharges_ebin5', 'TotalCharges_ebin10', 'TotalCharges_ebin20', 'TotalCharges_round10', 'Charge_Difference_qbin5', 'Charge_Difference_qb

...

Epoch 1/256: val 1-auc_ovr = 0.088338
Epoch 2/256: val 1-auc_ovr = 0.087505
Epoch 3/256: val 1-auc_ovr = 0.088826
Epoch 4/256: val 1-auc_ovr = 0.088771
Epoch 5/256: val 1-auc_ovr = 0.088329
Epoch 6/256: val 1-auc_ovr = 0.088230
Epoch 7/256: val 1-auc_ovr = 0.092484
Epoch 8/256: val 1-auc_ovr = 0.088094
Epoch 9/256: val 1-auc_ovr = 0.087316
Epoch 10/256: val 1-auc_ovr = 0.088470
Epoch 11/256: val 1-auc_ovr = 0.087413
Epoch 12/256: val 1-auc_ovr = 0.087585
Epoch 13/256: val 1-auc_ovr = 0.085839
Epoch 14/256: val 1-auc_ovr = 0.086095
Epoch 15/256: val 1-auc_ovr = 0.084837
Epoch 16/256: val 1-auc_ovr = 0.084620
Epoch 17/256: val 1-auc_ovr = 0.084280
Epoch 18/256: val 1-auc_ovr = 0.084302
Epoch 19/256: val 1-auc_ovr = 0.084987
Epoch 20/256: val 1-auc_ovr = 0.085987
Epoch 21/256: val 1-auc_ovr = 0.085378
Epoch 22/256: val 1-auc_ovr = 0.086725
Epoch 23/256: val 1-auc_ovr = 0.086555
Epoch 24/256: val 1-auc_ovr = 0.086522
Epoch 25/256: val 1-auc_ovr = 0.088833
Epoch 26/256: val 1-auc_ovr = 0.08

`Trainer.fit` stopped: `max_epochs=256` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Obtained ensemble weights: [18  1 13  5]


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


--- Fold 5 AUC: 0.91559 ---

Fold Scores : [np.float64(0.91699), np.float64(0.91795), np.float64(0.91723), np.float64(0.91826), np.float64(0.91559)]
Mean CV AUC : 0.91721 ± 0.00093
Overall OOF AUC: 0.91708


In [8]:
pd.DataFrame({'oof_ensemble_td' : oof_preds}).to_csv('oof_preds_ensemble_td.csv', index=False)
pd.DataFrame({'test_ensemble_td': test_preds}).to_csv('test_preds_ensemble_td.csv', index=False)
#submission
sample = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv')
pd.DataFrame({'id': sample['id'], TARGET : test_preds}).to_csv('Submission_Ensemble_TD_Model.csv', index=False)